In [ ]:
import os
import sys
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

PROJECT_ROOT = os.path.abspath("..") 
sys.path.append(PROJECT_ROOT)

In [ ]:

from dataset.otto_trace import TraceOttoDataSet
from model.trace import TRACE
from utils.SplitData import split_data_Train_Val_Test
from utils.feature_engineering import get_between_features, get_elapsed_feature
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")



In [ ]:
best_checkpoint = torch.load("")

In [ ]:
trace_model = TRACE(
    num_embeddings_aid=1855603,
    num_embeddings_event_type=4,
    embedding_dim=32,
    num_classes=1 
)
trace_model = trace_model.to(device)

In [ ]:
trace_model.load_state_dict(best_checkpoint["model_state_dict"])

In [ ]:
dataset_processed  = TraceOttoDataSet(
    file_name='../train.jsonl',
    input_seq_len=64,
    min_timestamps_per_sample=16,
    max_samples=100
)

In [ ]:
_, _, test_loader = split_data_Train_Val_Test(dataset_processed, batch_size=32)

In [ ]:
trace_model.eval()
criterion = nn.BCEWithLogitsLoss()
test_loss = 0
test_correct = 0
test_total = 0
test_accuracy = 0 

with torch.no_grad():
    for inputs_test, targets_test in test_loader:
        label_test_task = targets_test["ATC"].unsqueeze(1)
        
        inputs_test = {k: v.to(device)  for k, v in inputs_test.items()}
        
        delta_elapsed = get_elapsed_feature(inputs_test["timestamps"]).to(device)
        delta_between = get_between_features(inputs_test["timestamps"]).to(device)
        
        logit_test = trace_model(
                    inputs_test["aid"],
                    inputs_test["type"],
                    delta_elapsed,
                    delta_between
                )
    
        loss_test = criterion(logit_test, label_test_task.float())
        
        test_loss += loss_test.item()
        
         #RA1 Debuggin
        probs_test = torch.sigmoid(logit_test)
        preds_test = (probs_test >= 0.5).float()
        test_correct += (preds_test == label_test_task).sum().item()
        
        test_total += label_test_task.numel()
        
    test_loss /= len(test_loader)
    test_accuracy = test_correct / max(test_total, 1)

    
    
    